# Phase 4: Forecast-Based Trading and Stochastic Adoption

Goal: turn the AR forecast into a tradeable rule, let agents adopt it via stochastic diffusion (equations 11 and 12), and watch what happens to the forecast as more agents act on it. Compares zero, slow, and fast diffusion regimes.

Reference: section 3.4 of the proposal for the AR rule, section 3.5 for stochastic diffusion, section 3.6 for the intra-period timing, and section 4.1 for rolling OOS R^2 and rolling effective phi. CLAUDE.md phase 4 done-when: at high adoption shares the rolling out-of-sample R^2 is visibly different from at low adoption shares, with the rolling effective AR coefficient moving in step.

In [ ]:
# Number of traders.
N = 200

# Number of periods. Long enough that both diffusion regimes can ramp from
# A=0 (at metric warm-up) to near saturation inside the observable window.
T = 8000

# Market-maker price-impact coefficient (equation 5).
mu = 0.05

# Reduced-form residual AR(1) coefficient in the return law (equation 6).
phi = 0.25

# Standard deviation of exogenous news shocks.
sigma_news = 0.01

# Standard deviation of individual null orders (equation 8).
sigma_q = 1.0

# Rolling AR window and order for the public forecast.
forecast_window = 250
forecast_p = 1

# Combined risk-aversion times perceived variance, the denominator in (1) and (3).
risk_scale = 0.001

# Per-trader position cap from equation (3). Kept modest so adopter impact is
# comparable to phi r_t in magnitude rather than dominating it.
q_cap = 0.05

# Trailing window for rolling forecast metrics and rolling effective phi.
eval_window = 1000

# Hold adoption at zero until the rolling metrics have just become finite,
# then release the diffusion. Without this delay, adoption ramps during the
# warm-up and the fast regime saturates before the first metric observation,
# leaving no low-adoption samples to compare against the high-adoption tail.
adoption_start_t = forecast_window + eval_window

# Seed shared by all regimes so demand and news shocks are paired across them.
seed = 71

# Output paths.
fig_dir = "../results/figures"
data_dir = "../results/data"

# Stochastic diffusion regimes (equation 11). Onset is held off until
# adoption_start_t = 1250, so each regime starts ramping from A=0 at the
# moment metrics first become finite. Slow then sweeps roughly A=0 to
# A=0.87 across the run, fast sweeps A=0 to saturation inside the same
# window.
regimes = [
    {"name": "zero adoption",   "pi": 0.0,    "delta": 0.0, "colour": "C0"},
    {"name": "slow diffusion",  "pi": 3e-4,   "delta": 0.0, "colour": "C1"},
    {"name": "fast diffusion",  "pi": 1e-3,   "delta": 0.0, "colour": "C3"},
]

In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import matplotlib.pyplot as plt

from reflexive_market import simulate
from reflexive_market.metrics import rolling_oos_r2, rolling_phi

In [ ]:
results = []
for regime in regimes:
    rng = np.random.default_rng(seed)
    out = simulate.run(
        T=T, N=N, mu=mu, phi=phi,
        sigma_news=sigma_news, sigma_q=sigma_q,
        rng=rng,
        forecast_window=forecast_window, forecast_p=forecast_p,
        risk_scale=risk_scale, q_cap=q_cap,
        adoption_pi=regime["pi"], adoption_delta=regime["delta"],
        adoption_start_t=adoption_start_t,
    )
    # Residual return per section 3.6: the part of r_{t+1} that is not
    # the contemporaneous adopter demand impact. R^2 against this isolates
    # how well the forecast tracks the underlying AR + news component.
    residual = out["returns"] - mu * out["demand"]
    out["residual"] = residual
    out["rolling_oos_r2_realised"] = rolling_oos_r2(out["returns"], out["forecasts"], eval_window)
    out["rolling_oos_r2_residual"] = rolling_oos_r2(residual, out["forecasts"], eval_window)
    out["rolling_phi"] = rolling_phi(out["returns"], eval_window)
    results.append({**regime, **out})

In [ ]:
# Summarise low-adoption vs high-adoption performance per regime. The low
# window picks up a span where the rolling fit is past warm-up but adoption
# has barely moved; the high window picks up the latest period where
# adoption has saturated (or, for zero adoption, simply the late span).
warmup = forecast_window + eval_window
low_lo, low_hi = warmup, warmup + 1500
high_lo, high_hi = T - 1500, T

print(f"input phi:  {phi:+.4f}    risk_scale: {risk_scale:.4f}    q_cap: {q_cap:.4f}")
print()
header = (
    f"{'regime':<18}{'A_low':>7}{'A_high':>9}"
    f"{'R2real_lo':>11}{'R2real_hi':>11}"
    f"{'R2resid_lo':>12}{'R2resid_hi':>12}"
    f"{'phi_lo':>9}{'phi_hi':>9}"
)
print(header)
for r in results:
    A_low = float(np.nanmean(r["adoption_share"][low_lo:low_hi]))
    A_high = float(np.nanmean(r["adoption_share"][high_lo:high_hi]))
    R2r_low = float(np.nanmean(r["rolling_oos_r2_realised"][low_lo:low_hi]))
    R2r_high = float(np.nanmean(r["rolling_oos_r2_realised"][high_lo:high_hi]))
    R2x_low = float(np.nanmean(r["rolling_oos_r2_residual"][low_lo:low_hi]))
    R2x_high = float(np.nanmean(r["rolling_oos_r2_residual"][high_lo:high_hi]))
    phi_low = float(np.nanmean(r["rolling_phi"][low_lo:low_hi]))
    phi_high = float(np.nanmean(r["rolling_phi"][high_lo:high_hi]))
    r["summary"] = (A_low, A_high, R2r_low, R2r_high, R2x_low, R2x_high, phi_low, phi_high)
    print(
        f"{r['name']:<18}{A_low:>7.3f}{A_high:>9.3f}"
        f"{R2r_low:>11.4f}{R2r_high:>11.4f}"
        f"{R2x_low:>12.4f}{R2x_high:>12.4f}"
        f"{phi_low:>9.4f}{phi_high:>9.4f}"
    )

## Adoption trajectories

Each regime is a different per-period switch probability pi, with no exit. Adoption transitions are held off until t = forecast_window + eval_window so each regime starts ramping from A=0 at the moment rolling metrics first become finite. With pi=0 the share stays at zero; the slow rate produces a gentle climb sweeping roughly A=0 to A=0.87 over the run; the fast rate climbs sharply, reaching saturation around mid-run and giving the high-adoption tail.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
for r in results:
    ax.plot(r["adoption_share"], color=r["colour"], linewidth=1.2, label=r["name"])
ax.set_title("Adoption share across regimes")
ax.set_xlabel("Time period")
ax.set_ylabel("Share of adopters")
ax.set_ylim(-0.02, 1.02)
ax.legend()
fig.tight_layout()
fig.savefig(f"{fig_dir}/phase_04_adoption_share.png", dpi=150)
plt.show()

## Rolling OOS R^2 versus adoption share, two views

The proposal frames the same forecast in two ways and they pull in opposite directions when adoption rises.

**Realised return view (section 4.1).** R^2 of the forecast against the realised return r_{t+1}. As adopters trade in the direction of their forecast, contemporaneous demand impact mu D_t adds to r_{t+1} in the same direction the forecast pointed, so the rolling fit's R^2 against realised returns climbs with adoption. This is the "self-fulfilling" reading the proposal warns about in section 3.6.

**Residual return view (section 3.6).** R^2 of the same forecast against the residual r_{t+1} - mu D_t = phi r_t + sigma eps_{t+1}. The residual process has a fixed AR coefficient phi by construction, but the rolling fit picks up the amplified empirical phi_eff from realised returns and therefore over-predicts the residual. R^2 against the residual falls with adoption, and turns negative once the rolling fit overshoots phi by enough. This is the "self-eroding" reading.

Both quantities come from the same forecast and the same realised path; only the evaluation target differs.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)
for r in results:
    a = r["adoption_share"]
    yr = r["rolling_oos_r2_realised"]
    yx = r["rolling_oos_r2_residual"]
    mask_r = np.isfinite(yr)
    mask_x = np.isfinite(yx)
    axes[0].plot(a[mask_r], yr[mask_r], color=r["colour"], linewidth=0.8, alpha=0.7, label=r["name"])
    axes[1].plot(a[mask_x], yx[mask_x], color=r["colour"], linewidth=0.8, alpha=0.7, label=r["name"])
for ax, title in zip(axes, ["Realised return (section 4.1)", "Residual return (section 3.6)"]):
    ax.axhline(0.0, color="k", linewidth=0.5)
    ax.set_title(f"OOS R^2 vs adoption: {title}")
    ax.set_xlabel("Adoption share")
    ax.set_ylabel("Rolling OOS R^2")
    ax.legend()
fig.tight_layout()
fig.savefig(f"{fig_dir}/phase_04_oos_r2_vs_adoption.png", dpi=150)
plt.show()

## Rolling effective phi versus adoption share

Same x axis; y is now the rolling lag-1 autocorrelation of returns over the same trailing window. The proposal's mechanism (section 3.6) predicts that contemporaneous adopter trading reshapes the next-period predictable component, so the empirical AR coefficient should move when adoption rises.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for r in results:
    a = r["adoption_share"]
    y = r["rolling_phi"]
    mask = np.isfinite(y)
    ax.plot(a[mask], y[mask], color=r["colour"], linewidth=0.8, alpha=0.7, label=r["name"])
ax.axhline(phi, color="k", linewidth=0.5, linestyle="--", label=f"input phi = {phi}")
ax.set_title("Rolling effective phi versus adoption share")
ax.set_xlabel("Adoption share")
ax.set_ylabel("Rolling lag-1 autocorrelation")
ax.legend()
fig.tight_layout()
fig.savefig(f"{fig_dir}/phase_04_phi_vs_adoption.png", dpi=150)
plt.show()

## Monte Carlo across many seeds

The single-seed plots above are useful for intuition but each curve is one noisy realisation. The proposal's claim is about *expected* behaviour at each adoption share. The cells below repeat the three regimes across 100 paired-shock seeds and bin the post-warm-up rolling metrics by adoption share, then plot the bin mean with a +/-1 SD shaded band. Adoption share is itself a random variable per seed, so each (A_t, metric_t) pair is treated as one observation; pooling across seeds and time gives many observations per adoption-share bin.

In [ ]:
num_seeds_mc = 100
mc_base_seed = 1000  # distinct from the single-seed example seed=71

mc_data = {
    regime["name"]: {"A": [], "R2_real": [], "R2_resid": [], "phi": []}
    for regime in regimes
}

for s in range(num_seeds_mc):
    seed_s = mc_base_seed + s
    for regime in regimes:
        rng_s = np.random.default_rng(seed_s)
        out_s = simulate.run(
            T=T, N=N, mu=mu, phi=phi,
            sigma_news=sigma_news, sigma_q=sigma_q,
            rng=rng_s,
            forecast_window=forecast_window, forecast_p=forecast_p,
            risk_scale=risk_scale, q_cap=q_cap,
            adoption_pi=regime["pi"], adoption_delta=regime["delta"],
            adoption_start_t=adoption_start_t,
        )
        residual_s = out_s["returns"] - mu * out_s["demand"]
        r2r_s = rolling_oos_r2(out_s["returns"], out_s["forecasts"], eval_window)
        r2x_s = rolling_oos_r2(residual_s, out_s["forecasts"], eval_window)
        rp_s = rolling_phi(out_s["returns"], eval_window)
        mask = np.isfinite(r2r_s) & np.isfinite(r2x_s) & np.isfinite(rp_s)
        mc_data[regime["name"]]["A"].append(out_s["adoption_share"][mask])
        mc_data[regime["name"]]["R2_real"].append(r2r_s[mask])
        mc_data[regime["name"]]["R2_resid"].append(r2x_s[mask])
        mc_data[regime["name"]]["phi"].append(rp_s[mask])

for name in mc_data:
    for key in mc_data[name]:
        mc_data[name][key] = np.concatenate(mc_data[name][key])

print(f"{'regime':<22}{'observations':>13}{'A_min':>9}{'A_max':>9}")
for regime in regimes:
    a = mc_data[regime["name"]]["A"]
    print(f"{regime['name']:<22}{a.size:>13}{a.min():>9.3f}{a.max():>9.3f}")

### Mean OOS R^2 vs adoption share with +/-1 SD bands

Both panels overlay all three regimes. The line is the mean R^2 in each adoption-share bin across all seeds and time; the shaded band is +/-1 standard deviation. Bins with fewer than 10 observations are dropped (mostly an issue at the extreme right edge of the slow regime, which the run does not always reach in 8000 periods).

In [ ]:
n_bins = 30
bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])


def bin_stats(x, y, edges):
    """Mean and std of y in adoption-share bins of x, plus per-bin counts.

    np.digitize returns ``len(edges)`` for values exactly equal to the
    rightmost edge, so the raw indices have to be clipped into the valid
    bin range. Without this, saturated A = 1.0 observations are silently
    dropped from the final bin, which is most of the fast-diffusion tail.
    """
    inds = np.digitize(x, edges) - 1
    nb = len(edges) - 1
    inds = np.clip(inds, 0, nb - 1)
    means = np.full(nb, np.nan)
    stds = np.full(nb, np.nan)
    counts = np.zeros(nb, dtype=int)
    for i in range(nb):
        m = inds == i
        c = int(m.sum())
        counts[i] = c
        if c > 1:
            means[i] = y[m].mean()
            stds[i] = y[m].std()
    return means, stds, counts


fig, axes = plt.subplots(1, 2, figsize=(12, 4))
panels = [
    (axes[0], "R2_real", "Realised return (section 4.1)"),
    (axes[1], "R2_resid", "Residual return (section 3.6)"),
]
for regime in regimes:
    name = regime["name"]
    color = regime["colour"]
    A = mc_data[name]["A"]
    for ax, key, _ in panels:
        m, s, c = bin_stats(A, mc_data[name][key], bin_edges)
        valid = c > 10
        ax.plot(bin_centers[valid], m[valid], color=color, linewidth=1.5, label=name)
        ax.fill_between(bin_centers[valid],
                         m[valid] - s[valid],
                         m[valid] + s[valid],
                         color=color, alpha=0.2)

for ax, _, title in panels:
    ax.axhline(0.0, color="k", linewidth=0.5)
    ax.set_title(f"OOS R^2 vs adoption: {title}")
    ax.set_xlabel("Adoption share")
    ax.set_ylabel("Rolling OOS R^2")
    ax.legend()

fig.suptitle(f"Monte Carlo across {num_seeds_mc} seeds (mean +/- 1 SD)", y=1.02)
fig.tight_layout()
fig.savefig(f"{fig_dir}/phase_04_oos_r2_vs_adoption_mc.png", dpi=150, bbox_inches="tight")
plt.show()

### Mean effective phi vs adoption share with +/-1 SD band

Same binning, applied to the rolling lag-1 autocorrelation of returns. The dashed line is the input phi.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for regime in regimes:
    name = regime["name"]
    color = regime["colour"]
    A = mc_data[name]["A"]
    m, s, c = bin_stats(A, mc_data[name]["phi"], bin_edges)
    valid = c > 10
    ax.plot(bin_centers[valid], m[valid], color=color, linewidth=1.5, label=name)
    ax.fill_between(bin_centers[valid],
                     m[valid] - s[valid],
                     m[valid] + s[valid],
                     color=color, alpha=0.2)
ax.axhline(phi, color="k", linewidth=0.5, linestyle="--", label=f"input phi = {phi}")
ax.set_title(f"Rolling effective phi vs adoption (Monte Carlo, {num_seeds_mc} seeds)")
ax.set_xlabel("Adoption share")
ax.set_ylabel("Rolling lag-1 autocorrelation")
ax.legend()
fig.tight_layout()
fig.savefig(f"{fig_dir}/phase_04_phi_vs_adoption_mc.png", dpi=150)
plt.show()

## Save numeric summary

Per-regime headline scalars: mean adoption share, mean rolling OOS R^2, and mean rolling effective phi at low-adoption versus high-adoption windows. The full per-period traces are reproducible from the seed; only the headline table goes into the npz.

In [ ]:
summary = np.array([r["summary"] for r in results])
regime_pi = np.array([r["pi"] for r in results])
regime_delta = np.array([r["delta"] for r in results])
regime_names = np.array([r["name"] for r in results])

# Summary columns:
# 0: A_low  1: A_high  2: R2_realised_low  3: R2_realised_high
# 4: R2_residual_low  5: R2_residual_high  6: phi_low  7: phi_high
np.savez(
    f"{data_dir}/phase_04_stochastic_adoption.npz",
    summary=summary,
    regime_pi=regime_pi,
    regime_delta=regime_delta,
    regime_names=regime_names,
    phi_input=np.array(phi),
    mu=np.array(mu),
    risk_scale=np.array(risk_scale),
    q_cap=np.array(q_cap),
    forecast_window=np.array(forecast_window),
    eval_window=np.array(eval_window),
    T=np.array(T),
    N=np.array(N),
    seed=np.array(seed),
)

## Done when

- Adoption share trajectories match the three regimes (flat at zero, slow climb, fast saturation).
- Rolling OOS R^2 against realised returns is visibly higher at high adoption than at low adoption: the self-fulfilling channel from section 3.6.
- Rolling OOS R^2 against residual returns falls with adoption and goes negative at high adoption: the self-eroding channel from the same section.
- Rolling effective phi rises with adoption, since adopter demand reinforces the AR signal in realised returns.

The two R^2 plots show the same forecast paying off statistically against the realised return path while losing predictive power against what is actually predictable in the underlying process. Phase 6's threshold A* will pick up where one or both metrics cross zero.